In [1]:
import pandas as pd
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.linear_model import LinearRegression,Ridge,Lasso,LassoCV
from sklearn.model_selection import train_test_split

In [34]:
df=pd.read_csv("used_car_price_dataset_extended.csv")
df

,make_year,mileage_kmpl,engine_cc,fuel_type,owner_count,price_usd,brand,transmission,color,service_history,accidents_reported,insurance_valid
0,2001,8.17,4000,Petrol,4,8587.64,Chevrolet,Manual,White,NaN,0,No
1,2014,17.59,1500,Petrol,4,5943.50,Honda,Manual,Black,NaN,0,Yes
2,2023,18.09,2500,Diesel,5,9273.58,BMW,Automatic,Black,Full,1,Yes
3,2009,11.28,800,Petrol,1,6836.24,Hyundai,Manual,Blue,Full,0,Yes
4,2005,12.23,1000,Petrol,2,4625.79,Nissan,Automatic,Red,Full,0,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1996,15.25,1500,Petrol,2,5498.71,Hyundai,Manual,Silver,NaN,0,Yes
9996,1997,14.45,5000,Diesel,1,9588.31,Honda,Manual,Gray,Full,0,Yes
9997,2002,19.41,1000,Diesel,4,4557.10,Kia,Manual,Black,Full,0,Yes
9998,2005,12.95,4000,Petrol,5,7413.59,BMW,Automatic,Blue,Full,0,Yes


In [35]:
df['service_history']=df['service_history'].fillna("None")
df['make_year']=2026-df['make_year']

df['service_history']=df['service_history'].map({"Full":2,"Partial":1,"None":0})
df['insurance_valid']=df['insurance_valid'].map({"Yes":1,"No":0})


In [81]:
df_encoded=pd.get_dummies(
    df,
    columns=['transmission','fuel_type','color'],
    drop_first=True,
    dtype=int
)
df_encoded

,make_year,mileage_kmpl,engine_cc,owner_count,price_usd,brand,service_history,accidents_reported,insurance_valid,transmission_Manual,fuel_type_Electric,fuel_type_Petrol,color_Blue,color_Gray,color_Red,color_Silver,color_White
0,25,8.17,4000,4,8587.64,Chevrolet,0,0,0,1,0,1,0,0,0,0,1
1,12,17.59,1500,4,5943.50,Honda,0,0,1,1,0,1,0,0,0,0,0
2,3,18.09,2500,5,9273.58,BMW,2,1,1,0,0,0,0,0,0,0,0
3,17,11.28,800,1,6836.24,Hyundai,2,0,1,1,0,1,1,0,0,0,0
4,21,12.23,1000,2,4625.79,Nissan,2,0,1,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,30,15.25,1500,2,5498.71,Hyundai,0,0,1,1,0,1,0,0,0,1,0
9996,29,14.45,5000,1,9588.31,Honda,2,0,1,1,0,0,0,1,0,0,0
9997,24,19.41,1000,4,4557.10,Kia,2,0,1,1,0,0,0,0,0,0,0
9998,21,12.95,4000,5,7413.59,BMW,2,0,1,0,0,1,1,0,0,0,0


In [82]:
X = df_encoded.drop(['brand', 'price_usd'], axis=1)
y = df_encoded['price_usd']

In [83]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [102]:
lr_model=LinearRegression()
lr_model.fit(X_train,y_train)
ridge_model=Ridge(alpha=10).fit(X_train, y_train)

In [103]:
a = [
    0.0001, 0.0005, 0.001, 0.005,
    0.01, 0.02, 0.05, 0.1, 0.2, 0.3,
    0.5, 0.7, 1, 2, 3, 4, 5, 7, 10,
    15, 20, 25, 30, 40, 50, 75, 100,
    150, 200, 300, 500, 750, 1000
]
lasso_cv_model = LassoCV(
    alphas=a,
    cv=5,
    max_iter=1000,
    random_state=42
)
lasso_cv_model.fit(X_train, y_train)
print(lasso_cv_model.alpha_)

7.0


In [104]:
y_pred=lr_model.predict(X_test)

In [105]:
MAE=mean_absolute_error(y_test,y_pred)
MSE=mean_squared_error(y_test,y_pred)
r2=r2_score(y_test,y_pred)

In [106]:
print("MAE",MAE)
print("MSE",MSE)
print("R2 SCORE",r2)

MAE 791.8544958753081
MSE 983788.9594460438
R2 SCORE 0.8763388593487744


In [107]:
print("Linear:", lr_model.score(X_train, y_train), lr_model.score(X_test, y_test))
print("Ridge:", ridge_model.score(X_train, y_train), ridge_model.score(X_test, y_test))
print("Lasso:", lasso_cv_model.score(X_train, y_train), lasso_cv_model.score(X_test, y_test))

Linear: 0.8708012209072555 0.8763388593487744
Ridge: 0.8707790248590712 0.8762682656273358
Lasso: 0.8706004381981121 0.876493086357146
